In [ ]:
from pathlib import Path

# Edit if your folder structure is different
PROJECT_ROOT = Path('../../').resolve()
DATA_DIR = PROJECT_ROOT / 'data'
TEST_CSV = DATA_DIR / 'proba_test_72.csv'

print(f'Using test prediction file: {TEST_CSV}')


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc


In [ ]:
df = pd.read_csv(TEST_CSV)
required_cols = {'death_72h'}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f'Missing required columns: {missing}')

if 'proba_mean' in df.columns:
    y_score = df['proba_mean'].to_numpy()
    score_col = 'proba_mean'
else:
    run_cols = [c for c in df.columns if c.startswith('proba_run')]
    if not run_cols:
        raise ValueError('No proba_mean or proba_run* columns found in test CSV.')
    y_score = df[run_cols].mean(axis=1).to_numpy()
    score_col = f'mean({len(run_cols)} runs)'

y_true = df['death_72h'].to_numpy().astype(int)
fpr, tpr, _ = roc_curve(y_true, y_score)
auroc = auc(fpr, tpr)

print(f'N = {len(df)}')
print(f'Positive rate (death_72h) = {y_true.mean():.4f}')
print(f'Prediction column = {score_col}')
print(f'AUROC = {auroc:.4f}')


In [ ]:
plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, label=f'Fusion Model (AUROC={auroc:.4f})', linewidth=2)
plt.plot([0, 1], [0, 1], linestyle='--', linewidth=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Main Cohort Fusion Model ROC Curve')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()
